# Task 1: NAV Trend Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
category = pd.read_csv("../data/raw/05_category_inflows.csv")
folio = pd.read_csv("../data/raw/06_industry_folio_count.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")
nav = pd.read_csv("../data/raw/02_nav_history.csv")

In [ ]:
nav_history["date"] = pd.to_datetime(nav_history["date"])
nav_analysis = nav_history.merge(fund_master[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left")

print("Number of Schemes:", nav_analysis["scheme_name"].nunique())
print("Total Records:", len(nav_analysis))

In [ ]:
fig = px.line(
    nav_analysis,
    x="date",
    y="nav",
    color="scheme_name",
    title="Daily NAV Trend of All Mutual Fund Schemes(2022–2026)"
)

fig.update_layout(
    height=700,
    xaxis_title="Date",
    yaxis_title="NAV",
    legend_title="Scheme"
)
fig.show()

In [ ]:
fig = px.line(
    nav_analysis,
    x="date",
    y="nav",
    color="scheme_name",
    title="NAV Trends with Key Market Phases"
)
fig.add_vrect(
    x0="2023-01-01",
    x1="2023-12-31",
    fillcolor="green",
    opacity=0.15,
    line_width=0,
    annotation_text="2023 Bull Run"
)
fig.add_vrect(
    x0="2024-01-01",
    x1="2024-12-31",
    fillcolor="red",
    opacity=0.12,
    line_width=0,
    annotation_text="2024 Market Correction"
)

fig.update_layout(
    height=700,
    xaxis_title="Date",
    yaxis_title="NAV"
)

fig.show()

# Task 2: AUM Growth Analysis

In [ ]:
aum["date"] = pd.to_datetime(aum["date"])
aum["year"] = aum["date"].dt.year
aum.head()

In [ ]:
print(aum["fund_house"].unique())

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(
    data=aum,
    x="year",
    y="aum_lakh_crore",
    hue="fund_house"
)
plt.title("AUM Growth by Fund House(2022–2025)")
plt.xlabel("Year")
plt.ylabel("AUM (Lakh Crore)")
plt.legend(title="Fund House", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
aum.sort_values("aum_lakh_crore",ascending=False).head(10)

In [ ]:
sbi_aum = aum[aum["fund_house"] == "SBI Mutual Fund"]
plt.figure(figsize=(10, 6))
sns.barplot(
    data=sbi_aum,
    x="year",
    y="aum_lakh_crore"
)
plt.title("SBI Mutual fund AUM Growth")
plt.xlabel("Year")
plt.ylabel("AUM")

plt.annotate(
    "₹12.5 Lakh Cr",
    xy=(3, 12.5),
    xytext=(2.5, 13),
    arrowprops=dict(arrowstyle="->")
)
plt.show()

# Task 3: SIP Inflow Trend Analysis

In [ ]:
sip["month"] = pd.to_datetime(sip["month"])
sip.info()

In [ ]:
sip.loc[sip["sip_inflow_crore"].idxmax()]

In [ ]:
fig = px.line(
    sip,
    x="month",
    y="sip_inflow_crore",
    markers=True,
    title="Monthly SIP Inflows(2022–2025)"
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="SIP Inflow(Crore)",
    height=600
)
fig.show()

In [ ]:
max_row = sip.loc[sip["sip_inflow_crore"].idxmax()]
fig = px.line(
    sip,
    x="month",
    y="sip_inflow_crore",
    markers=True,
    title="Monthly SIP Inflows with All Time High"
)
fig.add_annotation(
    x=max_row["month"],
    y=max_row["sip_inflow_crore"],
    text=f"All-Time High<br>₹{max_row['sip_inflow_crore']:,} Cr",
    showarrow=True,
    arrowhead=2
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="SIP Inflow (Crore)",
    height=600
)
fig.show()

# Task 4: Category Inflow Heatmap

In [ ]:
category["month"] = pd.to_datetime(category["month"])
print("Shape:", category.shape)
category.head()

In [ ]:
heatmap_data = category.pivot(index="category",columns="month",values="net_inflow_crore")
heatmap_data.head()

In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data,cmap="RdYlGn",center=0,linewidths=0.3)
plt.title("Monthly Category-wise Net Inflows")
plt.xlabel("Month")
plt.ylabel("Fund Category")
plt.tight_layout()
plt.show()

In [ ]:
top_categories = (category.groupby("category")["net_inflow_crore"].sum().sort_values(ascending=False))
top_categories

# Task 5: Investor Demographics Analysis

In [ ]:
print("Dataset Shape:", transactions.shape)
print("\nAge Groups:")
print(transactions["age_group"].unique())
print("\nGender Categories:")
print(transactions["gender"].unique())
transactions.head()

In [ ]:
## Age Group Distribution
age_distribution = (transactions["age_group"].value_counts())
age_distribution

In [ ]:
fig = px.pie(values=age_distribution.values,
    names=age_distribution.index,
    title="Investor Distribution by Age Group"
)
fig.show()

In [ ]:
## SIP Amount Analysis by Age Group
sip_transactions = transactions[transactions["transaction_type"] == "SIP"]
print("Total SIP Transactions:", len(sip_transactions))
sip_transactions.head()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=sip_transactions,x="age_group",y="amount_inr")
plt.title("SIP Amount Distribution by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Investment Amount (₹)")
plt.tight_layout()
plt.show()

In [ ]:
## Gender Distribution
gender_distribution = (transactions["gender"].value_counts())
gender_distribution

In [ ]:
fig = px.pie(values=gender_distribution.values,
    names=gender_distribution.index,
    title="Gender Distribution of Investors"
)
fig.show()

# Task 6: Geographic Distribution Analysis

In [ ]:
print("No of States:", transactions["state"].nunique())
print("\nCity Tier Categories:")
print(transactions["city_tier"].unique())
transactions.head()

In [ ]:
state_sip = (transactions[transactions["transaction_type"] == "SIP"].groupby("state")["amount_inr"]
    .sum()
    .sort_values(ascending=False)
)
state_sip.head(10)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x=state_sip.values,y=state_sip.index)
plt.title("State-wise SIP Investment Amount")
plt.xlabel("Total SIP Amount (₹)")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
city_tier_distribution = (transactions["city_tier"].value_counts())
city_tier_distribution

In [ ]:
fig = px.pie(values=city_tier_distribution.values,
    names=city_tier_distribution.index,
    title="T30 vs B30 City Tier Distribution"
)
fig.show()

In [ ]:
city_tier_percentage = (transactions["city_tier"].value_counts(normalize=True)* 100)
city_tier_percentage.round(2)

# Task 7: Industry Folio Growth Analysis

In [ ]:
folio["month"] = pd.to_datetime(folio["month"])
print("Shape:", folio.shape)
folio.head()


In [ ]:
start_folios = folio.iloc[0]["total_folios_crore"]
end_folios = folio.iloc[-1]["total_folios_crore"]
growth_pct = ((end_folios - start_folios) / start_folios) * 100

print(f"Starting Folios : {start_folios:.2f} Cr")
print(f"Ending Folios   : {end_folios:.2f} Cr")
print(f"Growth (%)      : {growth_pct:.2f}%")

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(folio["month"],folio["total_folios_crore"],marker="o",linewidth=2)
plt.title("Growth in Mutual Fund Folios (2022–2025)")
plt.xlabel("Year")
plt.ylabel("Total Folios (Crore)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
milestones = []
for target in [15, 20, 25]:
    result = folio[folio["total_folios_crore"] >= target]

    if not result.empty:
        milestones.append(result.iloc[0])

milestones = pd.DataFrame(milestones)
milestones[["month", "total_folios_crore"]]

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(folio["month"],folio["total_folios_crore"],marker="o",linewidth=2
)
for _, row in milestones.iterrows():
    plt.annotate(
        f"{row['total_folios_crore']:.2f} Cr",
        (row["month"], row["total_folios_crore"]),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center"
    )
plt.title("Mutual Fund Folio Growth with Key Milestones")
plt.xlabel("Year")
plt.ylabel("Total Folios (Crore)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Task 8: NAV Return Correlation Analysis

In [ ]:
selected_funds = (fund_master["amfi_code"].drop_duplicates().sample(10, random_state=42))

In [ ]:
nav_subset = nav[nav["amfi_code"].isin(selected_funds)].copy()
nav_subset["date"] = pd.to_datetime(nav_subset["date"])
print("Shape:", nav_subset.shape)
nav_subset.head()

In [ ]:
nav_pivot = nav_subset.pivot(index="date",columns="amfi_code",values="nav")
nav_pivot.head()

In [ ]:
daily_returns = (nav_pivot.pct_change().dropna())
daily_returns.head()

In [ ]:
correlation_matrix = (daily_returns.corr())
correlation_matrix.round(2)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix,annot=True,fmt=".2f",cmap="coolwarm",center=0)
plt.title("Daily Return Correlation Matrix (Selected Funds)")
plt.tight_layout()
plt.show()

In [ ]:
corr_pairs = correlation_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs < 1]
top_correlations = (corr_pairs.sort_values(ascending=False).drop_duplicates()
    .head(10)
)
top_correlations

# Task 9: Sector Allocation Analysis

In [ ]:
print("Portfolio Holdings Shape:", holdings.shape)
print("\nNumber of Unique Sectors:")
print(holdings["sector"].nunique())
holdings.head()

In [ ]:
sector_weights = (holdings.groupby("sector")["weight_pct"].sum()
    .sort_values(ascending=False)
)
sector_weights

In [ ]:
top_sectors = sector_weights.head(10)
top_sectors

In [ ]:
sector_summary = pd.DataFrame({"Sector": top_sectors.index,"Total Weight (%)": top_sectors.values})
sector_summary

In [ ]:
fig = px.pie(values=top_sectors.values,names=top_sectors.index,
    hole=0.45,
    title="Sector Allocation Across Mutual Fund Portfolios"
)
fig.show()

In [ ]:
top3_weight = top_sectors.head(3).sum()
print(f"Combined Weight of Top 3 Sectors: {top3_weight:.2f}%")

In [ ]:
sector_rank = (sector_summary.reset_index(drop=True))
sector_rank.index += 1
sector_rank

## EDA Findings Summary
1. The analysis shows that most mutual fund schemes experienced steady NAV growth over time, reflecting positive long-term performance.
2. Periodic fluctuations in NAV values indicate the influence of changing market conditions and economic events.
3. A small number of fund houses managed a significant portion of total assets, demonstrating strong investor confidence.
4. SIP contributions displayed a consistent upward trend, highlighting increasing interest in disciplined investing.
5. Equity-focused schemes attracted substantial investor inflows due to their growth potential.
6. Mutual fund participation was highest among working-age individuals, making them a key investor segment.
7. Both male and female investors actively contributed to mutual fund investments, showing widespread adoption.
8. Investment activity varied across states, reflecting regional differences in financial awareness and investment preferences.
9. Growth in investments from B30 regions suggests expanding mutual fund reach beyond major urban centers.
10. The continuous rise in folio counts indicates a growing investor base and sustained expansion of the mutual fund industry.